# Assistant biomédical à verdict calibré — Jour 4
### Génération d'explication (LLM), tests et analyse d'erreurs

Ce notebook suppose que le **Jour 3** a été exécuté : modèle calibré, seuil
d'abstention choisi, index FAISS et détecteur hors périmètre construits.

Objectifs du jour :
1. Intégrer un LLM génératif (zero-shot, prompté) pour transformer verdict +
   phrase citée en explication en langage naturel
2. Garantir que le LLM **explique** mais ne **décide jamais** du verdict
3. Faire tourner le pipeline complet sur le test held-out et analyser les erreurs


## 1. Setup

In [ ]:
!pip install -q transformers peft sentence-transformers faiss-cpu scikit-learn accelerate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_DIR = "/content/drive/MyDrive/assistant_biomedical"
DATA_DIR = f"{PROJECT_DIR}/data"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
RESULTS_DIR = f"{PROJECT_DIR}/results"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


## 2. Rechargement du modèle classifieur (base + LoRA) et des artefacts du Jour 3

In [ ]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
ADAPTER_DIR = f"{CKPT_DIR}/pubmedbert_lora_adapter_final"

LABEL2ID = {"yes": 0, "no": 1, "maybe": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MAX_LENGTH = 384

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
clf_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
clf_model.to(device)
clf_model.eval()

calibration = load_json(f"{RESULTS_DIR}/day3_calibration.json")
abstention = load_json(f"{RESULTS_DIR}/day3_abstention.json")
ood_stats = load_json(f"{RESULTS_DIR}/day3_ood_detection.json")

learned_temperature = calibration["temperature"]
ABSTENTION_THRESHOLD = abstention["threshold"]
OOS_THRESHOLD = ood_stats["oos_threshold"]

print("Température :", learned_temperature)
print("Seuil d'abstention :", ABSTENTION_THRESHOLD)
print("Seuil hors périmètre :", OOS_THRESHOLD)


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
faiss_index = faiss.read_index(f"{CKPT_DIR}/biomedical_corpus.index")

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 5]

def cite_evidence_sentence(question, context, top_k=1):
    sentences = split_sentences(context)
    if not sentences:
        return [context]
    sentence_embeddings = embedder.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)
    question_embedding = embedder.encode([question], convert_to_tensor=True, normalize_embeddings=True)
    similarities = (sentence_embeddings @ question_embedding.T).squeeze(1)
    top_indices = torch.topk(similarities, k=min(top_k, len(sentences))).indices.tolist()
    return [sentences[i] for i in top_indices]

def ood_score(question, k=5):
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    similarities, _ = faiss_index.search(q_emb, k)
    return 1 - similarities.mean()

print("Fonctions du Jour 3 rechargées.")


## 3. Fonction verdict calibré (reprise du Jour 3, sans explication)

In [ ]:
@torch.no_grad()
def get_verdict(question, context):
    inputs = tokenizer(question, context, truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(device)
    logits = clf_model(**inputs).logits.cpu()
    probs = F.softmax(logits / learned_temperature, dim=1).squeeze(0)
    confidence, pred_id = torch.max(probs, dim=0)
    confidence, pred_id = confidence.item(), pred_id.item()

    if confidence < ABSTENTION_THRESHOLD:
        verdict = "incertain"
    else:
        verdict = ID2LABEL[pred_id]

    return {
        "verdict": verdict,
        "raw_label": ID2LABEL[pred_id],
        "confidence": round(confidence, 4),
    }


## 4. Chargement du LLM génératif pour l'explication

`Qwen2.5-1.5B-Instruct` : assez léger pour tourner confortablement sur T4 en
fp16, utilisé uniquement en **zero-shot prompté**, jamais fine-tuné. Il ne fait
que reformuler/expliquer un verdict déjà décidé par le classifieur.

In [ ]:
from transformers import AutoModelForCausalLM

EXPLAIN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

explain_tokenizer = AutoTokenizer.from_pretrained(EXPLAIN_MODEL_NAME)
explain_model = AutoModelForCausalLM.from_pretrained(
    EXPLAIN_MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
explain_model.eval()
print("LLM d'explication chargé.")


## 5. Prompt d'explication — garde-fous stricts

Règles imposées au prompt :
- Le LLM **ne doit jamais changer ou contredire le verdict** déjà décidé
- Il ne doit utiliser **que la phrase citée**, aucune connaissance externe
- Il doit rester concis (2-3 phrases) et rappeler qu'il ne s'agit pas d'un
  conseil médical
- Pour un verdict "incertain", il doit expliquer que la phrase citée ne permet
  pas de conclure clairement

In [ ]:
def build_explanation_prompt(question, citation, verdict):
    if verdict == "incertain":
        instruction = (
            "Le système ne peut pas conclure avec suffisamment de confiance. "
            "Explique en 2 phrases maximum pourquoi la phrase source ci-dessous "
            "ne permet pas de répondre clairement à la question, sans avancer "
            "toi-même un verdict."
        )
    else:
        verdict_fr = {"yes": "OUI", "no": "NON", "maybe": "INCERTAIN (peut-être)"}[verdict]
        instruction = (
            f"Le verdict déterminé est : {verdict_fr}. "
            "Explique en 2 phrases maximum, en te basant UNIQUEMENT sur la phrase "
            "source ci-dessous, pourquoi ce verdict est cohérent. Ne contredis "
            "jamais ce verdict et n'utilise aucune connaissance extérieure à la "
            "phrase source."
        )

    return (
        f"{instruction}\n\n"
        f"Question : {question}\n"
        f"Phrase source : {citation}\n\n"
        "Explication concise :"
    )

@torch.no_grad()
def generate_explanation(question, citation, verdict, max_new_tokens=80):
    prompt = build_explanation_prompt(question, citation, verdict)
    messages = [
        {"role": "system", "content": (
            "Tu es un assistant d'aide à la lecture biomédicale. Tu expliques des "
            "verdicts déjà établis, tu ne les modifies jamais, et tu ne donnes "
            "aucun conseil médical ni diagnostic."
        )},
        {"role": "user", "content": prompt},
    ]
    inputs = explain_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(explain_model.device)

    output = explain_model.generate(
        inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=explain_tokenizer.eos_token_id,
    )
    text = explain_tokenizer.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()
    return text


## 6. Pipeline complet : verdict + citation + explication + garde-fous

In [ ]:
DISCLAIMER = "Cet outil ne fournit pas de conseil médical ni de diagnostic."

def full_pipeline(question, context):
    # 1. Détection hors périmètre — coupe court avant tout le reste
    score = ood_score(question)
    if score > OOS_THRESHOLD:
        return {
            "in_scope": False,
            "message": "Question hors du périmètre biomédical couvert par cet assistant.",
        }

    # 2. Verdict calibré + abstention
    verdict_info = get_verdict(question, context)

    # 3. Citation de la phrase source
    citation = cite_evidence_sentence(question, context)[0]

    # 4. Explication en langage naturel (le LLM ne décide jamais du verdict)
    explanation = generate_explanation(question, citation, verdict_info["verdict"])

    return {
        "in_scope": True,
        "verdict": verdict_info["verdict"],
        "raw_label": verdict_info["raw_label"],
        "confidence": verdict_info["confidence"],
        "citation": citation,
        "explanation": explanation,
        "disclaimer": DISCLAIMER,
    }


In [ ]:
final_test = load_json(f"{DATA_DIR}/final_test_expert_holdout.json")

# Démonstration sur 3 exemples
for ex in final_test[:3]:
    result = full_pipeline(ex["question"], ex["context"])
    print("Question   :", ex["question"])
    print("Vrai label :", ex["label"])
    print("Résultat   :", json.dumps(result, ensure_ascii=False, indent=2))
    print("---")


## 7. Vérification de cohérence verdict / explication

Contrôle heuristique simple : l'explication ne doit pas affirmer littéralement
le verdict opposé. Ce n'est pas une garantie absolue, mais un filet de sécurité
qui signale les cas à revoir manuellement.

In [ ]:
CONTRADICTION_MARKERS = {
    "yes": ["ne confirme pas", "infirme", "réponse est non", "réponse : non"],
    "no": ["confirme que", "réponse est oui", "réponse : oui"],
    "maybe": [],  # verdict déjà incertain, peu de contradictions possibles
}

def contains_contradiction(explanation, verdict):
    if verdict not in CONTRADICTION_MARKERS:
        return False
    explanation_lower = explanation.lower()
    return any(marker in explanation_lower for marker in CONTRADICTION_MARKERS[verdict])


## 8. Exécution sur le test held-out (échantillon limité pour le temps Colab)

⚠️ La génération LLM est plus lente que la classification seule. Sur T4 gratuit,
on limite à ~200 exemples pour rester dans un temps raisonnable (~15-20 min).
Le classifieur seul, lui, a déjà été évalué en entier au Jour 2-3.

In [ ]:
EVAL_SAMPLE_SIZE = min(200, len(final_test))
eval_sample = final_test[:EVAL_SAMPLE_SIZE]

from tqdm import tqdm

pipeline_outputs = []
for ex in tqdm(eval_sample, desc="Pipeline complet (verdict + explication)"):
    result = full_pipeline(ex["question"], ex["context"])
    result["question"] = ex["question"]
    result["true_label"] = ex["label"]
    if result.get("in_scope"):
        result["contradiction_flag"] = contains_contradiction(result["explanation"], result["verdict"])
    pipeline_outputs.append(result)

save_json(pipeline_outputs, f"{RESULTS_DIR}/day4_pipeline_outputs.json")
print("Sorties du pipeline sauvegardées.")


## 9. Analyse d'erreurs

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

in_scope_outputs = [o for o in pipeline_outputs if o.get("in_scope")]

y_true = [o["true_label"] for o in in_scope_outputs]
y_pred = [o["verdict"] if o["verdict"] != "incertain" else "abstention" for o in in_scope_outputs]

labels_order = ["yes", "no", "maybe", "abstention"]
print(classification_report(y_true, y_pred, labels=[l for l in labels_order if l in set(y_true) | set(y_pred)], zero_division=0))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[l for l in labels_order if l in set(y_true) | set(y_pred)])
print("Matrice de confusion (lignes = vrai, colonnes = prédit) :")
print([l for l in labels_order if l in set(y_true) | set(y_pred)])
print(cm)


In [ ]:
# Taux d'abstention effectif sur cet échantillon
n_abstention = sum(1 for o in in_scope_outputs if o["verdict"] == "incertain")
print(f"Taux d'abstention sur l'échantillon évalué : {n_abstention}/{len(in_scope_outputs)} "
      f"({n_abstention/len(in_scope_outputs):.1%})")

# Taux de contradiction explication/verdict détectées
n_contradictions = sum(1 for o in in_scope_outputs if o.get("contradiction_flag"))
print(f"Contradictions détectées (heuristique) : {n_contradictions}/{len(in_scope_outputs)}")


In [ ]:
# Accuracy par tranche de confiance, pour visualiser où le modèle se trompe le plus
answered = [o for o in in_scope_outputs if o["verdict"] != "incertain"]

bins = [(0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
print(f"{'Confiance':>14s} {'N':>6s} {'Accuracy':>10s}")
for lo, hi in bins:
    subset = [o for o in answered if lo <= o["confidence"] < hi]
    if subset:
        acc = sum(1 for o in subset if o["verdict"] == o["true_label"]) / len(subset)
        print(f"[{lo:.1f}-{hi:.1f}) {len(subset):>6d} {acc:>10.4f}")


In [ ]:
# Exemples d'erreurs pour analyse qualitative
errors = [o for o in answered if o["verdict"] != o["true_label"]]
print(f"Nombre d'erreurs sur l'échantillon : {len(errors)}/{len(answered)}\n")

for o in errors[:5]:
    print("Question    :", o["question"])
    print("Vrai label  :", o["true_label"], "| Prédit :", o["verdict"], "| Confiance :", o["confidence"])
    print("Citation    :", o["citation"])
    print("Explication :", o["explanation"])
    print("---")


## 10. Focus sur la classe "maybe" (généralement la plus difficile)

In [ ]:
maybe_cases = [o for o in in_scope_outputs if o["true_label"] == "maybe"]
if maybe_cases:
    correct_maybe = sum(1 for o in maybe_cases if o["verdict"] == "maybe")
    abstained_maybe = sum(1 for o in maybe_cases if o["verdict"] == "incertain")
    print(f"Cas 'maybe' dans l'échantillon : {len(maybe_cases)}")
    print(f"  Correctement prédits 'maybe' : {correct_maybe}")
    print(f"  Résultat en abstention        : {abstained_maybe}")
    print(f"  Autres (erreur yes/no)        : {len(maybe_cases) - correct_maybe - abstained_maybe}")
else:
    print("Aucun cas 'maybe' dans cet échantillon limité — relancer sur un échantillon plus grand si besoin.")


In [ ]:
error_analysis_summary = {
    "sample_size": len(in_scope_outputs),
    "abstention_rate": n_abstention / len(in_scope_outputs),
    "contradiction_rate": n_contradictions / len(in_scope_outputs),
    "accuracy_answered": sum(1 for o in answered if o['verdict'] == o['true_label']) / len(answered) if answered else None,
}
save_json(error_analysis_summary, f"{RESULTS_DIR}/day4_error_analysis_summary.json")
print("Résumé de l'analyse d'erreurs sauvegardé.")


## 11. Bilan du Jour 4

- ✅ LLM génératif (Qwen2.5-1.5B-Instruct, zero-shot) intégré pour l'explication
- ✅ Garde-fous : le LLM explique mais ne décide jamais du verdict, contrôle
  heuristique de contradiction verdict/explication
- ✅ Pipeline complet testé de bout en bout (hors périmètre → verdict →
  abstention → citation → explication)
- ✅ Analyse d'erreurs : matrice de confusion, accuracy par tranche de
  confiance, focus sur la classe "maybe", exemples qualitatifs d'erreurs

**Prochaine étape (Jour 5)** : interface finale (Gradio/Streamlit), rédaction
de la documentation (README, data card), tests automatisés (pytest), et
répétition de la soutenance.